In [1]:
# Cell 1 - Import libraries and verify setup
import pandas as pd
import os

print("pandas version:", pd.__version__)
print("Setup complete!")

pandas version: 3.0.3
Setup complete!


In [2]:
# Cell 2 - Load state data first (smallest file, good starting point)
state_file = r"C:\Users\saahi\student_performance_project\data\ccd_sea_2425_state_nonfiscal_pub_elem_sec_ed_data.csv"

df_state = pd.read_csv(state_file, encoding='latin1', low_memory=False)

print("Shape:", df_state.shape)
print("\nColumns:", df_state.columns.tolist())
print("\nFirst 3 rows:")
df_state.head(3)

Shape: (14152, 12)

Columns: ['SCHOOL_YEAR', 'FIPST', 'STATENAME', 'ST', 'SEA_NAME', 'STATE_AGENCY_NO', 'GRADE', 'RACE_ETHNICITY', 'SEX', 'STUDENT_COUNT', 'TOTAL_INDICATOR', 'DMS_FLAG']

First 3 rows:


,SCHOOL_YEAR,FIPST,STATENAME,ST,SEA_NAME,STATE_AGENCY_NO,GRADE,RACE_ETHNICITY,SEX,STUDENT_COUNT,TOTAL_INDICATOR,DMS_FLAG
0,2024-2025,1,ALABAMA,AL,Alabama State Department of Education,1,Grade 1,American Indian or Alaska Native,Female,187.0,Category Set A - By Race/Ethnicity; Sex; Grade,Reported
1,2024-2025,1,ALABAMA,AL,Alabama State Department of Education,1,Grade 1,American Indian or Alaska Native,Male,191.0,Category Set A - By Race/Ethnicity; Sex; Grade,Reported
2,2024-2025,1,ALABAMA,AL,Alabama State Department of Education,1,Grade 1,Asian,Female,341.0,Category Set A - By Race/Ethnicity; Sex; Grade,Reported


In [8]:
# Cell 3 - Explore the state data
print("School years available:")
print(df_state['SCHOOL_YEAR'].unique())

print("\nStates (first 10):")
print(df_state['STATENAME'].unique()[:10])

print("\nGrades available:")
print(df_state['GRADE'].unique())

print("\nRace/Ethnicity categories:")
print(df_state['RACE_ETHNICITY'].unique())

print("\nMissing values per column:")
print(df_state.isnull().sum())

print("\n\nPercent missing per row:")
print(df_state.isnull().sum() / df_state.shape[0])

School years available:
<StringArray>
['2024-2025']
Length: 1, dtype: str

States (first 10):
<StringArray>
[             'ALABAMA',               'ALASKA',              'ARIZONA',
             'ARKANSAS',           'CALIFORNIA',             'COLORADO',
          'CONNECTICUT',             'DELAWARE', 'DISTRICT OF COLUMBIA',
              'FLORIDA']
Length: 10, dtype: str

Grades available:
<StringArray>
[          'Grade 1',          'Grade 10',          'Grade 11',
          'Grade 12',           'Grade 2',           'Grade 3',
           'Grade 4',           'Grade 5',           'Grade 6',
           'Grade 7',           'Grade 8',           'Grade 9',
      'Kindergarten',     'Not Specified',  'Pre-Kindergarten',
 'No Category Codes',          'Ungraded',   'Adult Education',
          'Grade 13']
Length: 19, dtype: str

Race/Ethnicity categories:
<StringArray>
[         'American Indian or Alaska Native',
                                     'Asian',
                 'Black or Af

Initial reactions to dataset:
* Only 339 missing student counts out of 14,152 rows - that's about 2.4%, very clean dataset
* TOTAL_INDICATOR is an important column I can use to filter
* Vague columns like No Category Codes or Not Specified can be dropped as they are not meaningful for the analysis


In [9]:
# Check what values are in TOTAL_INDICATOR
print(df_state['TOTAL_INDICATOR'].value_counts())

TOTAL_INDICATOR
Category Set A - By Race/Ethnicity; Sex; Grade                              12326
Subtotal 4 - By Grade                                                         874
Derived - Subtotal by Race/Ethnicity and Sex minus Adult Education Count      840
Derived - Education Unit Total minus Adult Education Count                     56
Education Unit Total                                                           56
Name: count, dtype: int64


In [10]:
# Cell 4 - Filter to Illinois and clean
df_illinois = df_state[
    (df_state['STATENAME'] == 'ILLINOIS') & 
    (df_state['TOTAL_INDICATOR'] == 'Category Set A - By Race/Ethnicity; Sex; Grade')
].copy()

# Drop rows where student count is missing
df_illinois = df_illinois.dropna(subset=['STUDENT_COUNT'])

# Drop catch-all category rows
df_illinois = df_illinois[~df_illinois['GRADE'].isin(['No Category Codes', 'Not Specified'])]
df_illinois = df_illinois[~df_illinois['RACE_ETHNICITY'].isin(['No Category Codes', 'Not Specified'])]

# Convert student count to integer
df_illinois['STUDENT_COUNT'] = df_illinois['STUDENT_COUNT'].astype(int)

print("Illinois rows:", len(df_illinois))
print("\nSample:")
df_illinois.head()

Illinois rows: 196

Sample:


,SCHOOL_YEAR,FIPST,STATENAME,ST,SEA_NAME,STATE_AGENCY_NO,GRADE,RACE_ETHNICITY,SEX,STUDENT_COUNT,TOTAL_INDICATOR,DMS_FLAG
3271,2024-2025,17,ILLINOIS,IL,Illinois State Board of Education,1,Grade 1,American Indian or Alaska Native,Female,125,Category Set A - By Race/Ethnicity; Sex; Grade,Reported
3272,2024-2025,17,ILLINOIS,IL,Illinois State Board of Education,1,Grade 1,American Indian or Alaska Native,Male,137,Category Set A - By Race/Ethnicity; Sex; Grade,Reported
3273,2024-2025,17,ILLINOIS,IL,Illinois State Board of Education,1,Grade 1,Asian,Female,3475,Category Set A - By Race/Ethnicity; Sex; Grade,Reported
3274,2024-2025,17,ILLINOIS,IL,Illinois State Board of Education,1,Grade 1,Asian,Male,3686,Category Set A - By Race/Ethnicity; Sex; Grade,Reported
3275,2024-2025,17,ILLINOIS,IL,Illinois State Board of Education,1,Grade 1,Black or African American,Female,10273,Category Set A - By Race/Ethnicity; Sex; Grade,Reported


In [ ]:
# Cell 5 - Save cleaned Illinois state data
output_path = r"C:\Users\saahi\student_performance_project\data\illinois_state_clean.csv"

df_illinois.to_csv(output_path, index=False)

print("Saved successfully!")
print("Rows:", len(df_illinois))
print("Columns:", df_illinois.columns.tolist())